<style>
.jp-RenderedText { font-size: 14pt; }
.jp-OutputArea-output { font-size: 14pt; }
</style>

## setup

In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path

import matplotlib as mpl
mpl.rcParams['font.size'] = 14

sys.path.insert(0, '.')
Path('plots').mkdir(exist_ok=True)

from src.features import sent_tokenize, word_tokenize, build_language_model, compute_h_lex, compute_h_syn
from src.baseline import evaluate_baseline
from src.full_system import evaluate_full_system

with open('data/processed/ham.json', encoding='utf-8') as f:
    ham = json.load(f)
with open('data/processed/phishing.json', encoding='utf-8') as f:
    phish = json.load(f)

print(len(ham), 'ham,', len(phish), 'phishing')

## tokeniser

In [2]:
normal_ex = "Thanks for the update. Could you send me the final version by Thursday? I need it before the meeting."
phishing_ex = "Your account has been suspended. Please verify your details immediately. Click the link below to restore access."

print('normal:')
print(' ', normal_ex)
print(' ', sent_tokenize(normal_ex))
print()
print('phishing:')
print(' ', phishing_ex)
print(' ', sent_tokenize(phishing_ex))

normal:
  Thanks for the update. Could you send me the final version by Thursday? I need it before the meeting.
  ['Thanks for the update.', 'Could you send me the final version by Thursday?', 'I need it before the meeting.']

phishing:
  Your account has been suspended. Please verify your details immediately. Click the link below to restore access.
  ['Your account has been suspended.', 'Please verify your details immediately.', 'Click the link below to restore access.']


## word tokeniser

In [3]:
print('normal:')
print(' ', word_tokenize(normal_ex))
print()
print('phishing:')
print(' ', word_tokenize(phishing_ex))

normal:
  ['Thanks', 'for', 'the', 'update', 'Could', 'you', 'send', 'me', 'the', 'final', 'version', 'by', 'Thursday', 'I', 'need', 'it', 'before', 'the', 'meeting']

phishing:
  ['Your', 'account', 'has', 'been', 'suspended', 'Please', 'verify', 'your', 'details', 'immediately', 'Click', 'the', 'link', 'below', 'to', 'restore', 'access']


## H_lex

In [4]:
lm = build_language_model(ham[:500])

sentences = [
    ('normal',   'Thanks for sending that over, I will review it before Friday.'),
    ('normal',   'The meeting has been moved to Thursday at two in the afternoon.'),
    ('normal',   'Let me know if you have any questions about the attached document.'),
    ('phishing', 'Please verify your account information immediately by clicking the link below.'),
    ('phishing', 'Your password expires in 24 hours. Update your details now to avoid suspension.'),
]

for label, sent in sentences:
    h = compute_h_lex(sent, lm)
    print(f'[{label}]  {round(h, 3) if h is not None else None}  {sent}')

[normal]  13.461  Thanks for sending that over, I will review it before Friday.
[normal]  13.812  The meeting has been moved to Thursday at two in the afternoon.
[normal]  12.51  Let me know if you have any questions about the attached document.
[phishing]  13.66  Please verify your account information immediately by clicking the link below.
[phishing]  13.764  Your password expires in 24 hours. Update your details now to avoid suspension.


## H_syn

In [5]:
for label, sent in sentences:
    h = compute_h_syn(sent)
    print(f'[{label}]  {round(h, 3) if h is not None else None}  {sent}')

[normal]  3.239  Thanks for sending that over, I will review it before Friday.
[normal]  2.815  The meeting has been moved to Thursday at two in the afternoon.
[normal]  3.239  Let me know if you have any questions about the attached document.
[phishing]  3.252  Please verify your account information immediately by clicking the link below.
[phishing]  3.374  Your password expires in 24 hours. Update your details now to avoid suspension.


## D2 baseline

In [6]:
baseline_results = evaluate_baseline(ham, phish)
print('D2 AUROC:', round(baseline_results['auroc'], 4))


=== D2 Baseline Evaluation (H_lex only) ===
  Train normal: 3096   Test normal: 775   Test phishing: 259
  Building n-gram language model on normal corpus...
  H_lex stats on normal data  — mean: 12.570  std: 1.550
  Anomaly threshold (mean + 2.0*std): 15.669

  AUROC: 0.8087

  Classification Report:
              precision    recall  f1-score   support

      Normal       0.82      1.00      0.90      7000
    Phishing       0.00      0.00      0.00      1559

    accuracy                           0.82      8559
   macro avg       0.41      0.50      0.45      8559
weighted avg       0.67      0.82      0.74      8559

D2 AUROC: 0.8087


## D3 full system

In [7]:
full_results = evaluate_full_system(ham, phish)
print('D3 AUROC:', round(full_results['auroc'], 4))


=== D3 Full System Evaluation (Joint H_lex + H_syn) ===
  Train: 2709 normal   Val: 387 normal + 162 phishing   Test: 775 normal + 259 phishing
  Building n-gram language model on normal corpus...
  Extracting (H_lex, H_syn) features from normal corpus...
  mu = [H_lex: 12.696, H_syn: 3.401]
  Covariance matrix:
[[1.495  0.3306]
 [0.3306 0.3769]]
  Anomaly threshold (tau): 1.4625

  AUROC: 0.7136

  Classification Report:
              precision    recall  f1-score   support

      Normal       0.93      0.57      0.71      6672
    Phishing       0.30      0.81      0.44      1546

    accuracy                           0.62      8218
   macro avg       0.62      0.69      0.58      8218
weighted avg       0.81      0.62      0.66      8218

D3 AUROC: 0.7136


## adversarial experiment

In [8]:
from src.adversarial import run_adversarial_experiment

adv = run_adversarial_experiment(baseline_results, full_results, phish, Path('plots'))

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



=== Adversarial Experiment: Synonym Substitution ===
  Substitution rate : 30% of eligible content words per sentence
  Phishing test emails: 259
  Applying synonym substitution...

  ADVERSARIAL EXPERIMENT RESULTS
  D2  H_lex only (baseline):
    AUROC original        : 0.8087
    AUROC after obfuscation: 0.8561
    Degradation           : -0.0473

  D3  Joint Mahalanobis (full system):
    AUROC original        : 0.7136
    AUROC after obfuscation: 0.7789
    Degradation           : -0.0653

  Finding: D2 degraded MORE than D3 under synonym substitution.
  Lexical signals are more vulnerable to synonym obfuscation than
  syntactic structure — consistent with the stated hypothesis.
  Saved: plots\adversarial_comparison.png


## plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import evaluate as ev

ev.plot_roc_comparison(baseline_results, full_results)
ev.plot_score_distributions(baseline_results, full_results)
ev.plot_feature_space(full_results)

for fname in ['roc_comparison.png', 'score_distributions.png', 'feature_space.png', 'adversarial_comparison.png']:
    img = mpimg.imread(f'plots/{fname}')
    plt.figure(figsize=(14, 9))
    plt.imshow(img)
    plt.axis('off')
    plt.show()